<a href="https://colab.research.google.com/github/B-21-g-23/AquaCropvisb/blob/main/CHIRPS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Colab: install Earth Engine API
!pip install --upgrade earthengine-api geemap

import ee
import geemap

# Trigger OAuth2 authentication flow
ee.Authenticate()
ee.Initialize(
    project = 'ee-birukgetaneh13',
    opt_url='https://earthengine-highvolume.googleapis.com')



In [ ]:
# --- Define Grid Points ---
grid_points = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point(39.697, 9.67), {'id': 'G1'}),
    ee.Feature(ee.Geometry.Point(39.45, 9.53), {'id': 'G2'}),
    ee.Feature(ee.Geometry.Point(39.31, 9.33), {'id': 'G5'}),
])

# --- Parameters ---
start_year = 1995
end_year = 2025
fallback_scale = 5000  # CHIRTS resolution ~0.05° (~5 km)

# --- Extraction Function ---
def extract_daily(image):
    """Extract daily max & min temperature values at grid points (°C)."""
    date = ee.Date(image.get('system:time_start')).format('YYYY-MM-dd')

    # Use new band names
    tmax_C = image.select('maximum_temperature').rename('tmax_C')
    tmin_C = image.select('minimum_temperature').rename('tmin_C')

    converted = image.addBands([tmax_C, tmin_C])

    sampled = converted.sampleRegions(
        collection=grid_points,
        scale=fallback_scale,
        geometries=True
    ).map(lambda f: f.set('date', date))

    return sampled

# --- Main Loop ---
all_data = ee.FeatureCollection([])

for year in range(start_year, end_year + 1):
    collection = (
        ee.ImageCollection("UCSB-CHG/CHIRTS/DAILY")
        .filterDate(f"{year}-01-01", f"{year+1}-01-01")
        .select(["maximum_temperature", "minimum_temperature"])  # updated
    )

    daily_fc = ee.FeatureCollection(
        ee.Algorithms.If(
            collection.size().gt(0),
            collection.map(extract_daily).flatten(),
            ee.FeatureCollection([])
        )
    )

    daily_fc = daily_fc.map(lambda f: f.set({'year': year}))
    all_data = all_data.merge(daily_fc)

# --- Export to Google Drive ---
task = ee.batch.Export.table.toDrive(
    collection=all_data,
    description=f'CHIRTS_tmax_tmin_gridpoints_{start_year}_{end_year}',
    folder='CHIRTStem',
    fileNamePrefix=f'CHIRTS_tmax_tmin_gridpoints_{start_year}_{end_year}',
    fileFormat='CSV'
)

task.start()
print(f"🚀 Export started for CHIRTS Tmax & Tmin {start_year}–{end_year}")

🚀 Export started for CHIRTS Tmax & Tmin 1995–2025
